# Research I — подготовка проверки гипотез H1–H3

**Назначение этого notebook:** собрать точный интерфейс существующего проекта перед написанием inference и анализа. Это **подготовительный этап, а не уже выполненная проверка гипотез**.

Из предоставленных материалов известны шесть вариантов моделей и каталог `artifacts/final_models/`. Приложенный код описывает HPO и вызов `run_final_benchmark`, но не содержит реализации загрузки checkpoint, dataset и preprocessing. Угадывать эти детали нельзя: такой эксперимент может работать технически, но сравнивать несовместимые предсказания.

Notebook читает исходники, небольшие JSON/CSV и исходный текст ячеек notebooks. Он не импортирует `satchange`, не загружает checkpoints через `torch.load`, не запускает модели, HPO или обучение. Исходные файлы проекта остаются неизменными. Запись выполняется только в **новый** каталог `artifacts/debug/hypotheses_preflight/`.

Результат — один ZIP для передачи в чат. В нём **нет весов, изображений датасета, `.git`, `.venv`, `.env` и выводов ячеек исходных notebooks**. Текст исходников и ячеек включается; перед передачей архива проверь, что в нём нет секретов, вписанных прямо в код.

## Что будет проверять экспериментальная версия

| Гипотеза | Сравнения | План измерений |
|---|---|---|
| **H1.** Потери качества Resized больше на мелких изменениях | Light Full ↔ Resized; Medium Full ↔ Resized; Heavy Full ↔ Resized | Recall по размерам размеченных областей; обнаружение областей; парные различия |
| **H2.** Возможное преимущество Heavy в pixel F1 связано с геометрией маски, а не только с нахождением новых областей | Heavy ↔ Light и Medium, отдельно при каждом разрешении | Pixel F1/IoU; ошибки границ и внутренних областей; matching областей; общие и уникальные обнаружения |
| **H3.** Взаимодополняемость сохраняется после согласования порогов | Пары архитектур, прежде всего внутри одного разрешения | PR-кривые; Jaccard TP и FP отдельно; уникальные TP; ΔRecall; Complementarity Gain; OR-объединение против одной модели |

Это гипотезы, а не ожидаемые выводы. У Heavy может не быть преимущества; Full может не иметь большего выигрыша на мелких областях.

**Согласуем перед test:** правила порогов, единый размер сетки для оценки, диапазоны размеров, определение обнаруженной области и статистические сравнения. Все настройки выбираются без подгонки по test. Если test уже использовался для предыдущего анализа, этот новый анализ необходимо обозначить как post-hoc, а не как независимое подтверждение.

## 1. Настройки

Запусти notebook из папки `notebooks/` первого исследования. Указан путь из твоего проекта. Если папка была переименована, измени только `PROJECT_ROOT`.

Набор seeds **не задаётся заранее**: будем читать фактически существующие checkpoints. `deployment_weights.pt` не считается дополнительным независимым seed.

In [1]:
from pathlib import Path

PROJECT_ROOT = Path(
    "../.."
).expanduser()

VARIANTS = (
    "light_full", "light_resized",
    "medium_full", "medium_resized",
    "heavy_full", "heavy_resized",
)

# Ограничения относятся только к текстовым файлам для архива.
MAX_TEXT_FILE_MIB = 8
MAX_TEXT_BUNDLE_MIB = 48
INCLUDE_RESEARCH_NOTEBOOK_SOURCE = True

# Не добавляй сюда .env, приватные ключи, веса или изображения.
EXTRA_TEXT_FILES = []

## 2. Безопасная подготовка

Проверка корня проекта; создание отдельной папки отчёта. Никакой установки пакетов или доступа к сети. Используется только стандартная библиотека Python.

In [2]:
import ast
import csv
import hashlib
import importlib.metadata
import io
import json
import os
import platform
import re
import subprocess
import sys
import uuid
import zipfile
from datetime import datetime, timezone

PROJECT_ROOT = PROJECT_ROOT.resolve(strict=True)
FINAL_ROOT = PROJECT_ROOT / "artifacts" / "final_models"
SOURCE_ROOT = PROJECT_ROOT / "src" / "satchange"
if not SOURCE_ROOT.is_dir() or not FINAL_ROOT.is_dir():
    raise RuntimeError(
        "Неверный PROJECT_ROOT: нужны src/satchange/ и artifacts/final_models/. "
        "Укажи корень Research I; ничего не изменено."
    )

MAX_TEXT_BYTES = int(MAX_TEXT_FILE_MIB * 1024**2)
MAX_BUNDLE_BYTES = int(MAX_TEXT_BUNDLE_MIB * 1024**2)
if MAX_TEXT_BYTES <= 0 or MAX_BUNDLE_BYTES <= 0:
    raise ValueError("Лимиты размера должны быть положительными.")

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
REPORT_PARENT = PROJECT_ROOT / "artifacts" / "debug" / "hypotheses_preflight"
# Не записываем через символические ссылки вне исходного проекта.
if not REPORT_PARENT.resolve().is_relative_to(PROJECT_ROOT):
    raise RuntimeError("Каталог отчёта разрешается за пределами PROJECT_ROOT.")
REPORT_DIR = REPORT_PARENT / run_id
REPORT_DIR.mkdir(parents=True, exist_ok=False)

issues = []
bundle = {}
manifest = []
secret_hits = []

# Это только проверка некоторых очевидных форматов, а не полный аудит секретов.
SECRET_PATTERN = re.compile(
    r"(?:github_pat_[A-Za-z0-9_]{20,}|gh[pousr]_[A-Za-z0-9]{20,}|"
    r"hf_[A-Za-z0-9]{20,}|sk-(?:proj-)?[A-Za-z0-9_-]{24,}|"
    r"AKIA[A-Z0-9]{16}|-----BEGIN (?:RSA |EC |OPENSSH )?PRIVATE KEY-----)"
)

def note(path, message):
    issues.append({"path": str(path), "message": message})

def record_bytes(archive_path, data, source=None):
    """Add text to the in-memory archive; never modify a project source file."""
    archive_path = str(archive_path).replace("\\", "/")
    ap = Path(archive_path)
    if ap.is_absolute() or ".." in ap.parts:
        raise ValueError(f"Unsafe archive path: {archive_path}")
    if archive_path in bundle:
        raise ValueError(f"Duplicate archive entry: {archive_path}")
    if len(data) > MAX_TEXT_BYTES:
        note(archive_path, "Skipped: text file exceeds per-file size limit; not truncated.")
        return False
    if b"\x00" in data:
        note(archive_path, "Skipped: NUL bytes; likely not a UTF-8 text file.")
        return False
    try:
        text = data.decode("utf-8-sig")
    except UnicodeDecodeError:
        note(archive_path, "Skipped: not UTF-8 text; not converted automatically.")
        return False
    for line_no, line in enumerate(text.splitlines(), 1):
        if SECRET_PATTERN.search(line):
            secret_hits.append({"file": archive_path, "line": line_no})
    if sum(map(len, bundle.values())) + len(data) > MAX_BUNDLE_BYTES:
        raise RuntimeError("Text bundle exceeds MAX_TEXT_BUNDLE_MIB; no archive was created.")
    bundle[archive_path] = data
    manifest.append({
        "archive_path": archive_path,
        "source_path": source,
        "bytes": len(data),
        "sha256": hashlib.sha256(data).hexdigest(),
    })
    return True

def read_project_text(path):
    """Read an ordinary UTF-8 file inside the project, with no symlink traversal."""
    path = Path(path)
    try:
        rel = path.relative_to(PROJECT_ROOT)
    except ValueError:
        note(path, "Skipped: outside project.")
        return None
    if any((PROJECT_ROOT / Path(*rel.parts[:i])).is_symlink()
           for i in range(1, len(rel.parts) + 1)):
        note(rel, "Skipped: symbolic link in path; inspect manually.")
        return None
    try:
        if not path.is_file():
            return None
        if path.stat().st_size > MAX_TEXT_BYTES:
            note(rel, "Skipped: file exceeds per-file size limit; not truncated.")
            return None
        data = path.read_bytes()
        data.decode("utf-8-sig")
        if b"\x00" in data:
            note(rel, "Skipped: contains NUL bytes.")
            return None
        return data
    except (OSError, UnicodeDecodeError) as exc:
        note(rel, f"Read failed: {type(exc).__name__}: {exc}")
        return None

def add_project_file(path):
    data = read_project_text(path)
    if data is None:
        return False
    rel = Path(path).relative_to(PROJECT_ROOT).as_posix()
    return record_bytes("project/" + rel, data, source=rel)

def write_json(path, value):
    Path(path).write_text(
        json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )

def git_read(*args):
    try:
        result = subprocess.run(
            ["git", "--no-optional-locks", "-C", str(PROJECT_ROOT), *args],
            capture_output=True, text=True, timeout=15, check=False,
        )
        if result.returncode == 0:
            return result.stdout.strip()
        return {"returncode": result.returncode, "stderr": result.stderr.strip()}
    except (OSError, subprocess.TimeoutExpired) as exc:
        return {"error": str(exc)}

print("Project:", PROJECT_ROOT)
print("Report:", REPORT_DIR)
print("Mode: read-only source inspection; no training, inference or torch.load.")

Project: .
Report: artifacts/debug/hypotheses_preflight/20261020T061436Z_133e34e0
Mode: read-only source inspection; no training, inference or torch.load.


## 3. Инвентаризация всех шести вариантов

Читаются только небольшие JSON и CSV. Для весов проверяются имена, размер и первые 200 байт: это позволяет заметить Git LFS pointer вместо скачанного файла. Содержимое тензоров не загружается.

Существование файла не доказывает совместимость checkpoint с текущим кодом — её проверит последующий inference smoke test.

In [3]:
EXPECTED_REPORTS = (
    "metadata.json", "benchmark.json", "validation_results.json",
    "final_test_results.json", "training_results.json",
)
WEIGHT_SUFFIXES = {".pt", ".pth", ".ckpt", ".safetensors", ".onnx"}
variant_inventory = []
metadata_details = {}

for variant in VARIANTS:
    folder = FINAL_ROOT / variant
    row = {"variant": variant, "folder_exists": folder.is_dir(),
           "deployment_file": "missing", "seed_files": [],
           "json_files": [], "missing_reports": [], "weights": []}
    metadata_details[variant] = {}
    if not folder.is_dir():
        note(variant, "Final model directory missing.")
        variant_inventory.append(row)
        continue

    for path in sorted(folder.glob("*.json")):
        data = read_project_text(path)
        if data is None:
            continue
        try:
            parsed = json.loads(data.decode("utf-8-sig"))
        except (ValueError, UnicodeDecodeError) as exc:
            note(path.relative_to(PROJECT_ROOT), f"Invalid JSON: {exc}")
            continue
        row["json_files"].append(path.name)
        metadata_details[variant][path.name] = {
            "root_type": type(parsed).__name__,
            "root_keys": sorted(parsed) if isinstance(parsed, dict) else None,
        }
        add_project_file(path)

    row["missing_reports"] = [name for name in EXPECTED_REPORTS
                              if not (folder / name).is_file()]
    for path in sorted(folder.iterdir()):
        if path.suffix.lower() not in WEIGHT_SUFFIXES or not path.is_file():
            continue
        item = {"file": path.name, "is_symlink": path.is_symlink()}
        try:
            item["bytes"] = path.stat().st_size
            if path.is_symlink():
                item["status"] = "symlink_not_read"
            else:
                with path.open("rb") as handle:
                    header = handle.read(200)
                item["status"] = (
                    "lfs_pointer" if header.startswith(b"version https://git-lfs.github.com/spec/v1")
                    else "empty" if item["bytes"] == 0
                    else "present_not_loaded"
                )
        except OSError as exc:
            item["status"] = "unreadable"
            item["error"] = str(exc)
        row["weights"].append(item)
        match = re.fullmatch(r"seed_(\d+)_best\.(?:pt|pth|ckpt)", path.name)
        if match:
            row["seed_files"].append(int(match.group(1)))
        if path.name == "deployment_weights.pt":
            row["deployment_file"] = item["status"]
    row["seed_files"].sort()
    variant_inventory.append(row)

for path in sorted(FINAL_ROOT.glob("*.csv")):
    add_project_file(path)

print(f"{'VARIANT':<18} {'DEPLOYMENT':<23} {'SEED FILES':<22} JSON")
print("-" * 80)
for row in variant_inventory:
    print(f"{row['variant']:<18} {row['deployment_file']:<23} "
          f"{str(row['seed_files']):<22} {len(row['json_files'])}")
    if row["missing_reports"]:
        print("  Missing reports:", ", ".join(row["missing_reports"]))

write_json(REPORT_DIR / "variant_inventory.json", variant_inventory)
write_json(REPORT_DIR / "metadata_schema.json", metadata_details)
print("\nSeed file names are inventory, not proof of independent training runs.")

VARIANT            DEPLOYMENT              SEED FILES             JSON
--------------------------------------------------------------------------------
light_full         present_not_loaded      [123, 777, 2024]       5
light_resized      present_not_loaded      [42, 123, 777]         5
medium_full        present_not_loaded      [42, 123, 777]         5
medium_resized     present_not_loaded      [42, 123, 777]         6
heavy_full         present_not_loaded      [42, 123, 777]         5
heavy_resized      present_not_loaded      [42]                   5

Seed file names are inventory, not proof of independent training runs.


## 4. Исходный код, конфигурации и версия среды

Включаются Python-файлы `src/` и `tests/`, конфигурации, небольшая документация и файлы зависимостей. Это позволит использовать **существующие** factory, dataset и preprocessing, а не создавать их предположительные копии.

Модули не импортируются. Список функций извлекается через разбор текста Python; это не запускает код.

In [4]:
SKIP_PARTS = {"__pycache__", ".ipynb_checkpoints", ".git", ".venv", "venv", "source_backups"}
selected = set()
for subdir, suffixes in (
    ("src", {".py"}),
    ("tests", {".py"}),
    ("configs", {".yaml", ".yml", ".json", ".toml"}),
    ("docs", {".md", ".rst", ".txt"}),
):
    folder = PROJECT_ROOT / subdir
    if folder.is_dir():
        for path in folder.rglob("*"):
            rel = path.relative_to(PROJECT_ROOT)
            if not SKIP_PARTS.intersection(rel.parts) and path.suffix.lower() in suffixes:
                selected.add(path)

for name in ("pyproject.toml", "setup.cfg", "setup.py", "README.md", ".gitattributes", ".gitignore"):
    path = PROJECT_ROOT / name
    if path.is_file():
        selected.add(path)
selected.update(PROJECT_ROOT.glob("requirements*.txt"))
for name in EXTRA_TEXT_FILES:
    path = PROJECT_ROOT / name
    if path.suffix.lower() not in {".py", ".md", ".txt", ".json", ".yaml", ".yml", ".toml", ".csv"}:
        raise ValueError(f"Disallowed extra file type: {name}")
    if path.name.startswith(".env"):
        raise ValueError("Environment files are not allowed in this archive.")
    selected.add(path)

symbols = {}
for path in sorted(selected):
    if not add_project_file(path):
        continue
    if path.suffix != ".py":
        continue
    rel = path.relative_to(PROJECT_ROOT).as_posix()
    text = bundle["project/" + rel].decode("utf-8-sig")
    try:
        tree = ast.parse(text, filename=rel)
        symbols[rel] = [
            {"kind": type(node).__name__, "name": node.name, "line": node.lineno,
             "signature": ast.unparse(node.args)
                          if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) else None}
            for node in tree.body
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef))
        ]
    except SyntaxError as exc:
        symbols[rel] = {"syntax_error": str(exc)}
        note(rel, "Could not parse Python source; raw text is still included.")

versions = {}
for package in ("torch", "torchvision", "numpy", "scipy", "pandas", "Pillow",
                "scikit-learn", "scikit-image", "timm", "transformers", "albumentations"):
    try:
        versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        versions[package] = None

runtime = {
    "python": sys.version,
    "python_executable": sys.executable,
    "platform": platform.platform(),
    "packages": versions,
    "project_root": str(PROJECT_ROOT),
    "git_head": git_read("rev-parse", "HEAD"),
    "git_branch": git_read("branch", "--show-current"),
    "git_tracked_status": git_read("status", "--porcelain", "--untracked-files=no"),
    "source_imported": False,
    "checkpoints_loaded": False,
    "gpu_used": False,
}
write_json(REPORT_DIR / "source_symbols.json", symbols)
write_json(REPORT_DIR / "runtime.json", runtime)
print("Source/config/documentation files selected:", len(selected))
print("Source/config/documentation files included:",
      sum(entry["archive_path"].startswith("project/") for entry in manifest))
print("Python:", platform.python_version())
print("torch version (not imported):", versions["torch"])
print("Git HEAD:", runtime["git_head"])

Source/config/documentation files selected: 60
Source/config/documentation files included: 93
Python: 3.11.6
torch version (not imported): 2.12.1+cu130
Git HEAD: 4c10f1c84190b67295ba01892f6cbdd1337376f9


## 5. Текст исходных notebooks и структура data/

Из notebooks сохраняются **только исходники code/markdown ячеек**, без outputs и attachments. Это важно: настройки benchmark иногда заданы в notebook, а не в metadata.

В `data/` читаются только имена каталогов и ограниченный список имён файлов. Изображения и маски не открываются и не копируются. Путь к реальному датасету дополнительно должен быть восстановлен из исходников/metadata.

In [5]:
notebook_index = []
if INCLUDE_RESEARCH_NOTEBOOK_SOURCE:
    notebooks_root = PROJECT_ROOT / "notebooks"
    for path in sorted(notebooks_root.rglob("*.ipynb")):
        rel = path.relative_to(PROJECT_ROOT)
        if SKIP_PARTS.intersection(rel.parts):
            continue
        lower = path.name.lower()
        if "hypothes" in lower or "tiny" in lower or "second" in lower or "semantic" in lower:
            continue
        if not any(token in lower for token in ("fc_siam", "bit", "mit", "heavy", "benchmark")):
            continue
        raw = read_project_text(path)
        if raw is None:
            continue
        try:
            original = json.loads(raw.decode("utf-8-sig"))
            code_parts, markdown_parts, cell_info = [], [], []
            for index, cell in enumerate(original.get("cells", [])):
                source = cell.get("source", "")
                text = "".join(source) if isinstance(source, list) else str(source)
                cell_type = cell.get("cell_type", "unknown")
                cell_info.append({"index": index, "type": cell_type, "source_characters": len(text)})
                if cell_type == "code":
                    code_parts.append(f"# %% Original cell {index}\n{text}\n")
                elif cell_type == "markdown":
                    markdown_parts.append(f"<!-- Original cell {index} -->\n{text}\n")
            base = "notebooks_source/" + rel.relative_to("notebooks").as_posix()
            record_bytes(base + ".code.txt", "\n".join(code_parts).encode("utf-8"), source=rel.as_posix())
            record_bytes(base + ".markdown.md", "\n\n".join(markdown_parts).encode("utf-8"), source=rel.as_posix())
            notebook_index.append({"file": rel.as_posix(), "cells": cell_info,
                                   "outputs_included": False, "attachments_included": False})
        except (ValueError, TypeError, AttributeError) as exc:
            note(rel, f"Notebook extraction failed: {exc}")

# Ограниченный обзор дерева: не читаем байты изображений и не следуем symlinks.
data_tree = []
MAX_TREE_ROWS = 250
MAX_FILES_PER_DIRECTORY = 3

def inspect_directory(folder, depth=0):
    if len(data_tree) >= MAX_TREE_ROWS or depth > 3:
        return
    try:
        entries = sorted(folder.iterdir(), key=lambda p: p.name)
    except OSError as exc:
        note(folder, f"Directory listing failed: {exc}")
        return
    files_shown = 0
    for path in entries:
        if len(data_tree) >= MAX_TREE_ROWS:
            break
        if path.is_symlink():
            kind = "symlink_not_followed"
        elif path.is_dir():
            kind = "directory"
        else:
            files_shown += 1
            if files_shown > MAX_FILES_PER_DIRECTORY:
                continue
            kind = "file_name_only"
        data_tree.append({"path": path.relative_to(PROJECT_ROOT).as_posix(), "kind": kind})
        if kind == "directory":
            inspect_directory(path, depth + 1)

if (PROJECT_ROOT / "data").is_symlink():
    data_tree.append({"path": "data", "kind": "symlink_not_followed"})
elif (PROJECT_ROOT / "data").is_dir():
    inspect_directory(PROJECT_ROOT / "data")

write_json(REPORT_DIR / "notebook_index.json", notebook_index)
write_json(REPORT_DIR / "data_directory_sample.json", data_tree)
print("Research notebooks extracted:", len(notebook_index))
for item in notebook_index:
    print(" ", item["file"])
print("Data tree entries (names only):", len(data_tree))

Research notebooks extracted: 9
  notebooks/01_fc_siam_diff.ipynb
  notebooks/02_fc_siam_diff_full_hpo.ipynb
  notebooks/03_bit_full.ipynb
  notebooks/03_bit_resized.ipynb
  notebooks/04_mit_full.ipynb
  notebooks/04_mit_resized.ipynb
  notebooks/Heavy_Full_HPO_v3_metadata_fix.ipynb
  notebooks/heavy_full_staged_resume.ipynb
  notebooks/heavy_resized_staged_resume.ipynb
Data tree entries (names only): 96


## 6. Сохранение плана экспериментов

План ниже будет уточнён после чтения фактической реализации evaluation. Он не заменяет проверку совместимости и воспроизводимости.

Особенно важно: **связная область бинарной маски — это область изменения, не гарантированно отдельное здание**. Взаимодополняемость по пикселям и по областям нужно считать и описывать отдельно.

In [6]:
EXPERIMENT_PLAN = """# Research I — H1/H2/H3 experiment specification (draft)

Status: preflight only. No hypothesis results have been computed.

## Shared protocol
- Models: light_full, light_resized, medium_full, medium_resized, heavy_full, heavy_resized.
- Use the existing project model factory, checkpoint loader and evaluation preprocessing after reviewing their source.
- Do not start HPO, fine-tuning or training. Never change final_models or historical protocols.
- Verify validation reproduction under the ORIGINAL evaluation protocol before any new analysis.
- Preserve that historical benchmark separately from a new common-grid analysis.
- For Full vs Resized, compare on the same ORIGINAL annotation grid with an explicitly documented score-upsample/threshold order. Do not compare native 256 labels against native 1024 labels as if the evaluation targets were identical.
- Select thresholds and size-bin cutoffs on validation (or training where appropriate), freeze them, then evaluate on test. Report achieved test recall; do not retune it to force a match.
- If test has already informed these hypotheses, describe the analysis as post-hoc/exploratory.
- Inspect real seeds; do not assume 42/123/777 exist for every variant. A deployment checkpoint is not an independent replicate of its source seed.
- Cache aligned scores using checkpoint, source, preprocessing and dataset-manifest fingerprints. Detect stale caches rather than silently reuse them.
- Keep image IDs and pair ordering. Stream masks/scores instead of holding all six full-resolution models' outputs in GPU memory.
- Validate on a small smoke batch before running a complete split. Reuse existing benchmark/cost reports; do not benchmark training again.

## H1: size-dependent Full/Resized difference
- Three paired comparisons: Full vs Resized within Light, Medium and Heavy.
- Define size from original-resolution ground-truth connected components, with explicit connectivity and treatment of border-truncated components.
- Derive bins on validation, then freeze. Report areas and sample counts; do not invent missing building instance IDs.
- Measure positive-pixel recall in each GT size group, and component detection separately with a fixed matching protocol.
- Do not report a size-group precision without an explicit rule for assigning false positives to that group.
- Main contrast: (Recall_full - Recall_resized)_small - (Recall_full - Recall_resized)_large.
- Report confidence intervals for the contrast, not just separate significance labels.
- These comparisons describe the trained pipelines; differing HPO, augmentation and seeds can confound a causal resolution claim.

## H2: geometry versus additional detections
- Compare Heavy vs Light and Medium at the same input-resolution condition.
- Report micro pixel F1/IoU plus boundary/interior error counts on the same output grid.
- Predefine boundary tolerance(s) in original pixels; inspect small components that have no interior after erosion.
- Use an explicit one-to-one component matching rule for detection precision/recall; report its IoU threshold and split/merge cases.
- Matching itself depends on shape quality, so include a complementary GT coverage analysis and do not call coverage alone full object-detection accuracy.
- Compare boundary/shape quality on common detected components as well as all GT components; disclose the conditioning.
- Similar numbers of detections are insufficient: verify which GT components are shared, A-only, B-only and missed by both.
- No claim that geometry explains Heavy's advantage if Heavy has no advantage, or if evidence contradicts that explanation.

## H3: threshold-controlled error complementarity
- Primary architecture pairs within each resolution; cross-resolution comparisons are separately labelled.
- Use validation PR curves to choose attainable operating points near predefined recall levels; retain achieved recall and precision.
- Matching recall does not imply matching precision, predictions or calibration.
- A_TP = prediction_A intersect GT; B_TP = prediction_B intersect GT.
- Report Jaccard_TP, Jaccard_FP and prediction-mask Jaccard with distinct labels; define empty-set cases explicitly.
- Report common TP, A-only TP, B-only TP, common FN and extra FP for the OR system.
- delta_recall = recall_OR - max(recall_A, recall_B).
- Best-recall-normalized CG = (TP_OR - max(TP_A, TP_B)) / (N_positive - max(TP_A, TP_B)).
- This CG is an operational definition for this study, not an assumption of a universal standard. If its denominator is zero, report undefined/NaN with a reason.
- Also report directed recovery of A's FN by B and vice versa, where denominators are nonzero.
- Compare OR against a SINGLE model whose threshold was selected on validation to target OR's validation recall, not just against the original single-model thresholds.
- Show extra false positives and saved computational benchmarks. Do not label the sum of individual latencies a measured ensemble latency.
- Persistence of complementarity after threshold alignment does not prove architecture causality; seeds and training can contribute.
- Zero CG does not prove thresholds caused the initial differences.

## Reliability and interpretation
- Paired resampling unit: image pair (or geographic cluster if known), not individual independent pixels.
- Distinguish image-sampling uncertainty from training-seed variation. One seed cannot estimate training variance.
- Report numerator/denominator counts and undefined metrics, not artificial zeros.
- Bootstrap results from a geographic dataset are conditional on the resampling assumptions; account for known spatial dependence.
- Freeze a small set of primary contrasts; mark additional plots/slices as exploratory.
- No automatic 'hypothesis proved' labels. Export evidence, effect sizes, limitations and selected examples for interpretation.

## Expected outputs of the experimental notebook
- preflight/reproduction_report.json
- protocol_frozen.json and sample/checkpoint/source manifests
- H1_size_results.csv and paired contrasts with intervals
- H2_geometry_component_results.csv and paired contrasts with intervals
- H3_pairwise_threshold_results.csv; PR curves; OR vs single-model comparison
- per_image_metrics.csv; per_component_metrics.csv; cache manifest
- a compact result archive WITHOUT weights, dataset images or the full score cache
"""
(REPORT_DIR / "EXPERIMENT_PLAN.md").write_text(EXPERIMENT_PLAN, encoding="utf-8")
print("Saved draft protocol:", REPORT_DIR / "EXPERIMENT_PLAN.md")

Saved draft protocol: artifacts/debug/hypotheses_preflight/20261020T061436Z_133e34e0/EXPERIMENT_PLAN.md


## 7. Создать архив контекста

В архив попадут только отобранные текстовые файлы и диагностические отчёты. Скрипт остановится, если обнаружит некоторые очевидные форматы токенов/приватных ключей; он не печатает сами секреты.

Архив содержит рабочие исходники проекта. Никакие файлы не отправляются автоматически. Если в списке предупреждений есть пропущенные ключевые исходники или metadata, передай предупреждения вместе с архивом.

In [7]:
write_json(REPORT_DIR / "issues.json", issues)
write_json(REPORT_DIR / "secret_scan_findings.json", secret_hits)

if secret_hits:
    print("Possible secrets found (values are NOT shown):")
    for hit in secret_hits:
        print(f"  {hit['file']}:{hit['line']}")
    raise RuntimeError(
        "Archive not created. Review these files locally. "
        "Do not send secret values in chat; prepare sanitized source and rerun."
    )

README = """# Context bundle for the LEVIR-CD hypothesis notebook

This is a preflight bundle, NOT model evaluation results.

- project/: selected project source/configuration and final-model JSON/CSV, with original bytes preserved.
- notebooks_source/: code and markdown cell sources only; no outputs or attachments.
- reports/: model inventory, source symbols, runtime and local Git state, directory-name samples and proposed experiment plan.
- manifest.json: hashes and sizes of included text entries.

No checkpoint tensors, raw dataset images, .env, .git, .venv or HPO run trees are included.
No project modules were imported. No model, training job or GPU benchmark was run.
Model compatibility and the exact preprocessing contract must be established by reading the included code.
A token-pattern scan is not a complete secret audit. The owner must review the bundle before sharing.
"""
record_bytes("README.md", README.encode("utf-8"))

for path in sorted(REPORT_DIR.iterdir()):
    if path.is_file() and path.suffix in {".json", ".md"}:
        record_bytes("reports/" + path.name, path.read_bytes())

# Не допускаем попадания очевидных секретов из новых отчётов.
if secret_hits:
    write_json(REPORT_DIR / "secret_scan_findings.json", secret_hits)
    raise RuntimeError("Possible secret found in report; archive not created. Check the local findings JSON.")

write_json(REPORT_DIR / "manifest.json", manifest)
archive_path = REPORT_DIR / "levir_hypotheses_context.zip"
# Mode 'x' prevents accidentally overwriting an earlier archive.
with zipfile.ZipFile(archive_path, mode="x", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for name, data in sorted(bundle.items()):
        zf.writestr(name, data)
    zf.writestr("manifest.json", json.dumps(manifest, ensure_ascii=False, indent=2).encode("utf-8"))

with zipfile.ZipFile(archive_path, "r") as zf:
    corrupt_entry = zf.testzip()
    if corrupt_entry is not None:
        raise RuntimeError(f"ZIP verification failed: {corrupt_entry}")
    for name in zf.namelist():
        if Path(name).suffix.lower() in WEIGHT_SUFFIXES or any(
            p in {".git", ".venv", "hyperparameter_search"} for p in Path(name).parts
        ):
            raise RuntimeError(f"Unexpected prohibited archive entry: {name}")

archive_hash = hashlib.sha256(archive_path.read_bytes()).hexdigest()
(REPORT_DIR / "archive_sha256.txt").write_text(archive_hash + "\n", encoding="utf-8")
print("Archive:", archive_path)
print(f"Size: {archive_path.stat().st_size / 1024**2:.2f} MiB")
print("Included text files:", len(manifest))
print("Warnings:", len(issues))
for item in issues[:20]:
    print(" -", item["path"], "—", item["message"])
if len(issues) > 20:
    print("See reports/issues.json for the remaining warnings.")
print("\nSend levir_hypotheses_context.zip after reviewing it.")
print("The actual H1/H2/H3 inference notebook will be built from these interfaces.")

Archive: artifacts/debug/hypotheses_preflight/20261020T061436Z_133e34e0/levir_hypotheses_context.zip
Size: 0.28 MiB
Included text files: 121
Warnings: 0

Send levir_hypotheses_context.zip after reviewing it.
The actual H1/H2/H3 inference notebook will be built from these interfaces.


## Дальнейший шаг

Передай `levir_hypotheses_context.zip` из напечатанного каталога. Дополнительно загружать 3.7 GB весов или исходные спутниковые снимки для адаптации кода не нужно.

По содержимому архива можно будет реализовать один экспериментальный notebook для всех шести вариантов: сначала проверка загрузки и воспроизведения validation, затем кеширование scores, фиксация протокола, H1–H3 и экспорт результатов.

**Никакие гипотезы этим подготовительным notebook ещё не проверены.** Запуск на настоящих моделях будет выполняться на твоём сервере после проверки интерфейсов.